# Stiffness and Applications

An equation is **stiff** when it mixes very fast and very slow processes. Explicit solvers are then forced into tiny steps to stay stable, long after the fast process has finished. The notebook starts with two classic stiff problems, then works through four applications from economics, epidemiology and pharmacology.

In [ ]:
Pollis.packages("OrdinaryDiffEq", "ForwardDiff", "Plots")

In [ ]:
using OrdinaryDiffEq, LinearAlgebra, ForwardDiff, Plots

## Robertson's chemical reaction

Three species react on time scales from $10^{-4}$ to $10^{4}$. The rate constants 0.04, $10^4$ and $3 \times 10^7$ differ by nine orders of magnitude.

In [ ]:
function rober(u, p, t)
    y1, y2, y3 = u
    return [-0.04y1 + 1e4 * y2 * y3, 0.04y1 - 1e4 * y2 * y3 - 3e7 * y2^2, 3e7 * y2^2]
end
prob = ODEProblem(rober, [1.0, 0.0, 0.0], (0.0, 1e5))
implicit = solve(prob, Rodas5P(); reltol=1e-8, abstol=1e-10)
explicit = solve(prob, Tsit5(); maxiters=200_000)
[(solver = "Rodas5P", steps = length(implicit.t) - 1, reached_t = implicit.t[end], retcode = implicit.retcode),
 (solver = "Tsit5", steps = length(explicit.t) - 1, reached_t = explicit.t[end], retcode = explicit.retcode)]

In [ ]:
ts = 10 .^ range(-6, 5, length=400)
plot(ts, [[implicit(t)[1] for t in ts] [1e4 * implicit(t)[2] for t in ts] [implicit(t)[3] for t in ts]], xscale=:log10, lw=2,
     label=["y1" "y2 x 10^4" "y3"], xlabel="t (log scale)", title="Robertson: fast start, slow finish")

The Jacobian's eigenvalues show why: once the fast transient is over, one eigenvalue sits near $-10^4$ while the solution changes on a time scale of thousands. An explicit solver's step is capped at about $3/10^4$.

In [ ]:
lam = eigvals(ForwardDiff.jacobian(u -> rober(u, nothing, 0.0), implicit(100.0)))
(eigenvalues_at_t_100 = lam, explicit_step_limit = 3 / maximum(abs.(lam)))

## The Van der Pol oscillator

Van der Pol found this equation in 1920s vacuum-tube circuits: $x'' - \mu(1 - x^2)x' + x = 0$. For large $\mu$ the solution creeps slowly and then jumps suddenly, a **relaxation oscillation** with period close to $(3 - 2\ln 2)\,\mu$. This is the Second-Order Equation tab's kind of problem at its hardest.

In [ ]:
mu = 1000.0
vdp(u, p, t) = [u[2], mu * (1 - u[1]^2) * u[2] - u[1]]
osc = solve(ODEProblem(vdp, [2.0, 0.0], (0.0, 5000.0)), Rodas5P(); reltol=1e-8, abstol=1e-10)
tt = 0:0.5:5000; x = [osc(t)[1] for t in tt]
ups = [tt[i] for i in 2:length(tt) if x[i - 1] < 0 <= x[i]]
display(plot(tt, x, lw=2, label="", xlabel="t", ylabel="x", title="Van der Pol, mu = $mu"))
(steps = length(osc.t) - 1, measured_period = diff(ups), asymptotic_period = (3 - 2log(2)) * mu)

## Economics: the Solow growth model

Capital per worker $k$ grows by saving $s k^\alpha$ and shrinks by depreciation and population growth $(\delta + n)k$. The steady state is $k^* = (s/(\delta + n))^{1/(1 - \alpha)}$. Near it, deviations shrink at rate $(1 - \alpha)(\delta + n)$, which gives the model's famous **speed of convergence**.

In [ ]:
s, alpha, delta, n = 0.25, 0.33, 0.06, 0.01
solow(k, p, t) = s * k^alpha - (delta + n) * k
kstar = (s / (delta + n))^(1 / (1 - alpha))
sol = solve(ODEProblem(solow, 1.0, (0.0, 150.0)), Tsit5())
plot(sol, lw=2, label="capital per worker", xlabel="years"); hline!([kstar], ls=:dash, label="steady state")

In [ ]:
rate = (1 - alpha) * (delta + n)
(steady_state = kstar, convergence_rate = rate, half_life_years = log(2) / rate, gap_closed_after_15_years = (sol(15.0) - 1) / (kstar - 1))

## Epidemiology: the SIR model

Kermack and McKendrick (1927): susceptibles $S$ are infected at rate $\beta S I$ and the infected recover at rate $\gamma$. The basic reproduction number is $R_0 = \beta/\gamma$. Two formulas can be checked against the simulation. The peak share infected is $1 - (1 + \ln(R_0 S_0))/R_0$ when $I_0$ is small. The share never infected solves $S_\infty = S_0 e^{-R_0(1 - S_\infty)}$.

In [ ]:
beta, gamma = 0.3, 0.1
R0 = beta / gamma
sir(u, p, t) = [-beta * u[1] * u[2], beta * u[1] * u[2] - gamma * u[2], gamma * u[2]]
sol = solve(ODEProblem(sir, [0.999, 0.001, 0.0], (0.0, 200.0)), Tsit5(); reltol=1e-8, abstol=1e-10)
display(plot(sol, lw=2, label=["susceptible" "infected" "recovered"], xlabel="days"))
tt = 0:0.1:200; infected = [sol(t)[2] for t in tt]
never_infected = foldl((x, _) -> 0.999 * exp(-R0 * (1 - x)), 1:200; init=0.5)   # fixed-point iteration
(peak_infected = maximum(infected), peak_formula = 1 - (1 + log(R0 * 0.999)) / R0, peak_day = tt[argmax(infected)],
 never_infected_simulated = sol(200.0)[1], never_infected_formula = never_infected)

## Pharmacology: a drug taken by mouth

An oral dose $a$ in the gut is absorbed at rate $k_a$ into the blood, where the amount $c$ is eliminated at rate $k_e$:

$$a' = -k_a a, \qquad c' = k_a a - k_e c.$$

The solution is the Bateman function, and the blood level peaks at $t_{\max} = \ln(k_a/k_e)/(k_a - k_e)$.

In [ ]:
ka, ke, dose = 1.2, 0.25, 100.0   # per hour, per hour, mg
pk(u, p, t) = [-ka * u[1], ka * u[1] - ke * u[2]]
sol = solve(ODEProblem(pk, [dose, 0.0], (0.0, 24.0)), Tsit5(); reltol=1e-10, abstol=1e-12)
bateman(t) = dose * ka / (ka - ke) * (exp(-ke * t) - exp(-ka * t))
tt = 0:0.001:24; blood = [sol(t)[2] for t in tt]
display(plot(sol, lw=2, label=["in the gut" "in the blood"], xlabel="hours", ylabel="mg"))
(tmax_formula = log(ka / ke) / (ka - ke), tmax_simulated = tt[argmax(blood)], largest_error_vs_bateman = maximum(abs.(blood .- bateman.(tt))))

## Exercises

1. Solve Robertson with `AutoTsit5(Rosenbrock23())`. How many steps does it take, and when does it switch?
2. Raise the Solow saving rate to 0.35 at year 50. Solve in two stages with `remake`, and find the new steady state and half-life.
3. For SIR, find the smallest vaccinated share $v$ (start with $S_0 = 1 - v$) that stops the epidemic from growing at all. Compare with $1 - 1/R_0$.
4. Repeat the drug dose every 8 hours for three days. Use a `PresetTimeCallback` or solve in stages. What is the steady trough level?